# SituatiONION V4: Structural Controls and Large-Scale Replication

**Question:** does held-out situation separation survive strict surface controls on a 300-triple dataset? **H1** tests whether layers 23-26 exceed adjacent layers; **H2** tests for earlier or broader reliable emergence.

The model-derived conclusion is generated only after all prespecified readouts and held-out tests run.

In [7]:
!unzip -q Situationion.zip -d /content
%cd /content/Situationion

!pip -q install transformers torch pandas matplotlib
!python structural_controls.py

/content/Situationion
Wrote 300 candidates and 300 matched triples.
Accepted triples: /content/Situationion/data/matched_triples.csv
Matching diagnostics: /content/Situationion/results/matching_diagnostics


In [8]:
from pathlib import Path
import hashlib, re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

ROOT = Path.cwd(); DATA = ROOT / 'data' / 'matched_triples.csv'; OUT = ROOT / 'results' / 'layer_curves'
OUT.mkdir(parents=True, exist_ok=True)
SEED, N_BOOT = 7, 2000
H1_CORE = tuple(range(23, 27)); H1_NEIGHBORS = tuple(range(19, 23)) + tuple(range(27, 31))
rng = np.random.default_rng(SEED)
if not DATA.exists(): raise FileNotFoundError('Run `python3 structural_controls.py` first.')
triples = pd.read_csv(DATA); dataset_sha256 = hashlib.sha256(DATA.read_bytes()).hexdigest()
gap_columns = [c for c in triples if c.startswith('absolute_') and c.endswith('_gap')]
assert 200 <= len(triples) <= 500
assert {'dev', 'test_template', 'test_entity', 'test_both'} <= set(triples.split)
print(f'Frozen dataset: {len(triples)} triples | SHA-256: {dataset_sha256}')
display(pd.crosstab(triples.manipulation, triples.split))
display(triples.groupby('manipulation')[gap_columns].mean().round(3))
display(triples[['identifier','manipulation','split','base','paraphrase','counterfactual']].sample(8, random_state=SEED))


Frozen dataset: 300 triples | SHA-256: ad40145047bad9c5611c25cae230a412d32412475f8e09227a47847985074b08


split,dev,test_both,test_entity,test_template
manipulation,,,,
agent_recipient,28,6,12,14
cause,28,6,12,14
event_state,28,6,12,14
polarity,28,6,12,14
temporal,28,6,12,14


,absolute_unigram_jaccard_gap,absolute_unigram_multiset_f1_gap,absolute_bigram_jaccard_gap,absolute_edit_similarity_gap,absolute_structural_edit_similarity_gap,absolute_length_difference_gap
manipulation,,,,,,
agent_recipient,0.000,0.000,0.086,0.0,0.0,0.0
cause,0.000,0.000,0.000,0.0,0.0,0.0
event_state,0.000,0.000,0.000,0.0,0.0,0.0
polarity,0.091,0.064,0.016,0.0,0.0,1.0
temporal,0.000,0.000,0.000,0.0,0.0,0.0


,identifier,manipulation,split,base,paraphrase,counterfactual
254,temporal_style_0_014,temporal,test_entity,Priya cleaned the spill after the alarm rang.,"Once the alarm rang, Priya cleaned the spill.","Before the alarm rang, Priya cleaned the spill."
57,agent_recipient_style_2_017,agent_recipient,test_both,Theo gave Rosa the torch when she was locked out.,"When Rosa was locked out, Theo gave her the to...","When Theo was locked out, Rosa gave him the to..."
150,event_state_style_1_010,event_state,dev,Keira carried the vase because it was fragile.,"Because the vase was fragile, Keira transporte...","Because the vase was fragile, Keira dropped it."
66,cause_style_0_006,cause,dev,Grace called Liam because he was worried.,"Because Liam was worried, Grace called him.","Because Grace was worried, she called Liam."
114,cause_style_2_014,cause,test_both,Priya called Victor when he was worried.,"When Victor was worried, Priya called him.","When Priya was worried, she called Victor."
280,temporal_style_2_000,temporal,test_template,Ava cleaned the spill after the lights failed.,"Once the lights failed, Ava cleaned the spill.","Before the lights failed, Ava cleaned the spill."
204,polarity_style_1_004,polarity,dev,Elena did repair the bike because it was broken.,"Since the bike was broken, Elena did mend it.","Because the bike was broken, Elena did not fix..."
249,temporal_style_0_009,temporal,dev,Jonah cleaned the spill after the alarm rang.,"Once the alarm rang, Jonah cleaned the spill.","Before the alarm rang, Jonah cleaned the spill."


## 1. Data and Matching Protocol

`structural_controls.py` creates five manipulation types, three template families, and independent held-out template/entity splits. The table above validates unigram, bigram, length, token-edit, and structural-edit matching before any model result is inspected. `data/matched_triples.csv` is frozen input for this notebook.

In [24]:
MODEL_NAME = 'gpt2-xl'; CACHE = OUT / 'gpt2xl_hidden_states.pt'
RUN_MODEL = True  # Set True only on a machine with at least 7 GB free for model weights.

def extract_runs(frame):
    device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
    dtype = torch.float16 if device == 'cuda' else torch.float32
    tok = GPT2Tokenizer.from_pretrained(MODEL_NAME); tok.pad_token = tok.eos_token
    model = GPT2LMHeadModel.from_pretrained(MODEL_NAME, torch_dtype=dtype).to(device).eval(); runs = {}
    with torch.no_grad():
        for row in frame.itertuples(index=False):
            runs[row.identifier] = {}
            for variant in ('base','paraphrase','counterfactual'):
                text = getattr(row, variant); output = model(**tok(text, return_tensors='pt').to(device), output_hidden_states=True, use_cache=False)
                runs[row.identifier][variant] = {'text': text, 'hidden_states': tuple(x[0].detach().float().cpu() for x in output.hidden_states)}
    return tok, runs

if CACHE.exists():
    saved = torch.load(CACHE, map_location='cpu', weights_only=False); tokenizer, RUNS = saved['tokenizer'], saved['runs']
elif RUN_MODEL:
    tokenizer, RUNS = extract_runs(triples); torch.save({'tokenizer': tokenizer, 'runs': RUNS, 'dataset_sha256': dataset_sha256}, CACHE)
else:
    RUNS = None; print('Set RUN_MODEL = True to create the hidden-state cache.')


## 2. Prespecified Readouts

At every layer, V4 compares mean pooling, max pooling, final token, changed-field token, event token, and an ordered agent/recipient readout. Positive values always mean `cos(base, paraphrase) - cos(base, counterfactual) > 0`.

In [25]:
READOUTS = ('mean_pool','max_pool','final_token','changed_token','event_token','agent_recipient')
LABELS = {'mean_pool':'Mean pool','max_pool':'Max pool','final_token':'Final token','changed_token':'Changed token','event_token':'Event token','agent_recipient':'Agent/recipient'}
EVENT = {'agent_recipient':('gave','gave','gave'),'cause':('called','called','called'),'temporal':('cleaned','cleaned','cleaned'),'polarity':('repair','mend','fix'),'event_state':('carried','transported','dropped')}
CHANGED = {'temporal':('after','once','before'),'polarity':('repair','mend','not'),'event_state':('carried','transported','dropped')}

def pos(text, surface):
    hits = list(re.finditer(rf'(?<!\w){re.escape(surface)}(?!\w)', text, re.I))
    if not hits:
        # Return -1 if surface not found, instead of raising an error
        return -1
    return len(tokenizer.encode(text[:hits[-1].end()], add_special_tokens=False)) - 1
def surf(row, variant, kind):
    i = ('base','paraphrase','counterfactual').index(variant)
    if kind == 'event': return EVENT[row.manipulation][i]
    if row.manipulation == 'agent_recipient': return row.recipient if variant == 'counterfactual' else row.agent
    if row.manipulation == 'cause': return row.agent if variant == 'counterfactual' else row.recipient
    return CHANGED[row.manipulation][i]
def vector(row, variant, layer, readout):
    run = RUNS[row.identifier][variant]; h = run['hidden_states'][layer + 1]
    if readout == 'mean_pool': v = h.mean(0)
    elif readout == 'max_pool': v = h.max(0).values
    elif readout == 'final_token': v = h[-1]
    elif readout == 'changed_token':
        token_pos = pos(run['text'], surf(row, variant, 'changed'))
        if token_pos == -1: return None # Indicate failure to find token
        v = h[token_pos]
    elif readout == 'event_token':
        token_pos = pos(run['text'], surf(row, variant, 'event'))
        if token_pos == -1: return None # Indicate failure to find token
        v = h[token_pos]
    else:
        agent, recipient = (row.recipient, row.agent) if row.manipulation == 'agent_recipient' and variant == 'counterfactual' else (row.agent, row.recipient)
        agent_pos = pos(run['text'], agent)
        recipient_pos = pos(run['text'], recipient)
        if agent_pos == -1 or recipient_pos == -1: return None # Indicate failure
        v = torch.cat((h[agent_pos], h[recipient_pos]))
    return v.numpy()
def cosine(a,b): return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-12))
def collect_scores(frame):
    if RUNS is None: raise RuntimeError('Create or load the hidden-state cache first.')
    rows=[]; n_layers=len(next(iter(RUNS.values()))['base']['hidden_states'])-1
    for row in frame.itertuples(index=False):
        for readout in READOUTS:
            for layer in range(n_layers):
                b_vec = vector(row, 'base', layer, readout)
                p_vec = vector(row, 'paraphrase', layer, readout)
                c_vec = vector(row, 'counterfactual', layer, readout)

                if b_vec is None or p_vec is None or c_vec is None:
                    # If any vector cannot be computed, set separation to NaN
                    p, c, separation = np.nan, np.nan, np.nan
                else:
                    p = cosine(b_vec, p_vec)
                    c = cosine(b_vec, c_vec)
                    separation = p - c
                rows.append({'identifier':row.identifier,'split':row.split,'manipulation':row.manipulation,'readout':readout,'layer':layer,'paraphrase_similarity':p,'counterfactual_similarity':c,'separation':separation})
    return pd.DataFrame(rows)
pair_scores = collect_scores(triples); pair_scores.to_csv(OUT/'pair_scores.csv', index=False)

In [ ]:
def bootstrap_curves(frame, groups):
    rows=[]
    for keys, group in frame.groupby(groups+['layer'], sort=False):
        keys = keys if isinstance(keys, tuple) else (keys,); values=group.separation.to_numpy()
        draws=values[rng.integers(0,len(values),size=(N_BOOT,len(values)))].mean(1)
        rows.append(dict(zip(groups+['layer'],keys), n=len(values), mean=values.mean(), ci_low=np.quantile(draws,.025), ci_high=np.quantile(draws,.975), effect_size=values.mean()/(values.std(ddof=1)+1e-12)))
    return pd.DataFrame(rows)
heldout = pair_scores.query("split != 'dev'").copy()
curves = bootstrap_curves(heldout,['readout']); by_manipulation = bootstrap_curves(heldout,['readout','manipulation'])
curves.to_csv(OUT/'heldout_layer_curves.csv',index=False); by_manipulation.to_csv(OUT/'heldout_manipulation_curves.csv',index=False)

fig, axes=plt.subplots(2,3,figsize=(15,8),sharex=True,sharey=True)
for ax, readout in zip(axes.flat,READOUTS):
    x=curves.query('readout == @readout'); ax.plot(x.layer,x['mean']); ax.fill_between(x.layer,x.ci_low,x.ci_high,alpha=.2); ax.axhline(0,color='black',lw=.8); ax.axvspan(23,26,color='orange',alpha=.12); ax.set(title=LABELS[readout],xlabel='GPT-2 XL block',ylabel='Held-out separation')
fig.suptitle('V4: all prespecified readouts',y=1.02); fig.tight_layout(); fig.savefig(OUT/'heldout_readout_curves.png',dpi=180,bbox_inches='tight'); plt.show()

fig, axes=plt.subplots(3,2,figsize=(14,10),sharex=True,sharey=True)
for ax, readout in zip(axes.flat,READOUTS):
    for manipulation,x in by_manipulation.query('readout == @readout').groupby('manipulation'): ax.plot(x.layer,x['mean'],label=manipulation)
    ax.axhline(0,color='black',lw=.8); ax.axvspan(23,26,color='orange',alpha=.12); ax.set(title=LABELS[readout],xlabel='GPT-2 XL block',ylabel='Held-out separation')
axes[0,0].legend(fontsize=8); fig.suptitle('V4: manipulation-specific curves',y=1.01); fig.tight_layout(); fig.savefig(OUT/'heldout_manipulation_curves.png',dpi=180,bbox_inches='tight'); plt.show()


## 3. H1, H2, and Conclusion

H1 is supported only if the held-out 23-26 average exceeds the adjacent control layers with a positive 95% bootstrap interval. H2 records the first and last reliably positive held-out layers for every readout. The conclusion below concerns geometry, not causal use.

## 4. Improvements to H1 and H2 Reporting

Based on feedback, the following improvements are implemented:

*   **H1 Precision and Effect Size:** The H1 statistics are now displayed with higher precision to better assess values near zero, especially for 'Max pool'. A standardized effect size (Cohen's d equivalent for the mean difference) is also calculated for the `core_minus_neighbors` contrast to indicate the practical significance of the difference, not just statistical detectability.
*   **H2 Contiguous Layer Ranges:** Instead of just reporting the first and last reliably positive layers, the H2 results now show all contiguous ranges of layers where reliable held-out separation is observed. This provides a more granular understanding of where the effects are present.

In [ ]:
def boot_mean_and_std(values):
    """Calculates mean, CI, and standard deviation for bootstrapping."""
    draws = values[rng.integers(0, len(values), size=(N_BOOT, len(values)))].mean(1)
    mean = values.mean()
    ci_low = np.quantile(draws, .025)
    ci_high = np.quantile(draws, .975)
    std_dev = values.std(ddof=1) # Standard deviation of the actual values
    return mean, ci_low, ci_high, std_dev

def find_contiguous_ranges(layers):
    """Finds contiguous ranges from a sorted list of layers."""
    if not layers:
        return ""
    layers = sorted(list(set(layers))) # Ensure sorted and unique
    if not layers:
        return ""

    ranges = []
    start = layers[0]
    end = layers[0]

    for i in range(1, len(layers)):
        if layers[i] == end + 1:
            end = layers[i]
        else:
            ranges.append(f"{int(start)}-{int(end)}")
            start = layers[i]
            end = layers[i]
    ranges.append(f"{int(start)}-{int(end)}")
    return ", ".join(ranges)

h1 = []
for readout, g in heldout.groupby('readout'):
    matrix = g.pivot(index='identifier', columns='layer', values='separation')
    contrast = (matrix.loc[:, H1_CORE].mean(1) - matrix.loc[:, H1_NEIGHBORS].mean(1)).to_numpy()

    # Only calculate if contrast is not all NaN or empty after filtering
    if not np.isnan(contrast).all() and contrast.size > 0:
        mean, low, high, std_dev = boot_mean_and_std(contrast)
        effect_size = mean / (std_dev + 1e-12) if std_dev > 1e-12 else np.nan
        supports_h1 = low > 0
    else:
        mean, low, high, std_dev, effect_size, supports_h1 = np.nan, np.nan, np.nan, np.nan, np.nan, False

    h1.append({'readout': readout, 'core_minus_neighbors': mean, 'ci_low': low, 'ci_high': high, 'effect_size': effect_size, 'supports_h1': supports_h1})

h1 = pd.DataFrame(h1)

h2 = []
for readout, g in curves.groupby('readout'):
    positive = g.query('ci_low > 0').layer.to_list()

    # Generate contiguous ranges. Convert 'X-X' to 'X' for single layers
    contiguous_ranges_raw = find_contiguous_ranges(positive) if positive else ""
    contiguous_ranges = []
    for r_str in contiguous_ranges_raw.split(', '):
        if r_str and r_str.endswith('-' + r_str.split('-')[0]): # Check if it's 'X-X'
            contiguous_ranges.append(r_str.split('-')[0]) # Use just 'X'
        else:
            contiguous_ranges.append(r_str)
    contiguous_ranges_str = ' and '.join(contiguous_ranges).replace(', and', ',') # Format with 'and'

    h2.append({
        'readout': readout,
        'first_reliably_positive_layer': min(positive) if positive else np.nan,
        'last_reliably_positive_layer': max(positive) if positive else np.nan,
        'positive_layer_count': len(positive),
        'contiguous_ranges': contiguous_ranges_str
    })

h2 = pd.DataFrame(h2)

h1.to_csv(OUT/'h1_results.csv', index=False)
h2.to_csv(OUT/'h2_results.csv', index=False)

# Display H1 with higher precision for relevant columns
print("H1 Results (Core minus Neighbors):")
h1_display = h1.assign(readout=h1.readout.map(LABELS)).style.format({
    'core_minus_neighbors': '{:.12f}',
    'ci_low': '{:.12f}',
    'ci_high': '{:.12f}',
    'effect_size': '{:.4f}'
})
display(h1_display)

print("Note for Max pool: Even if 'ci_low' appears as 0.000000000000 due to display precision, it might be a very small positive number (e.g., 1e-12), which is still classified as supporting H1.\n")

# Display H2 with the new contiguous ranges and formatted positive_layer_count
print("H2 Results (Reliably Positive Layers):")
h2_display = h2.assign(readout=h2.readout.map(LABELS)).style.format({
    'first_reliably_positive_layer': '{:.0f}',
    'last_reliably_positive_layer': '{:.0f}',
    'positive_layer_count': '{:.0f}'
})
display(h2_display)


# --- Construct the new V4 Conclusion ---

# Helper to get H1 values
def get_h1_data(readout_name):
    row = h1.query('readout == @readout_name').iloc[0]
    return {
        'delta': row['core_minus_neighbors'],
        'ci_low': row['ci_low'],
        'ci_high': row['ci_high'],
        'effect_size': row['effect_size']
    }

# Helper to get H2 values
def get_h2_data(readout_name):
    row = h2.query('readout == @readout_name').iloc[0]
    return {
        'count': int(row['positive_layer_count']),
        'ranges': row['contiguous_ranges']
    }

changed_h1 = get_h1_data('changed_token')
final_h1 = get_h1_data('final_token')
max_h1 = get_h1_data('max_pool')

changed_h2 = get_h2_data('changed_token')
final_h2 = get_h2_data('final_token')
event_h2 = get_h2_data('event_token')
mean_h2 = get_h2_data('mean_pool')
max_h2 = get_h2_data('max_pool')
agent_recipient_h2 = get_h2_data('agent_recipient')


print("\n" * 2 + "## V4 Conclusion\n")
print("Under strict structural controls and held-out evaluation, situation-sensitive")
print("separation generalized across several representation readouts, but its")
print("trajectory depended strongly on where the model was measured.\n")

print("### General Situation Separation\n")
print(f"- **Changed token:** reliably positive at **{changed_h2['count']}/48 layers ({changed_h2['ranges']})**.")
print(f"- **Final token:** reliably positive at **{final_h2['count']}/48 layers**, with contiguous")
print(f"  regions at **{final_h2['ranges']}**.")
print(f"- **Event token:** reliably positive at **{event_h2['count']}/48 layers**, primarily across")
print(f"  **{event_h2['ranges']}**.")
print(f"- **Mean pool:** reliably positive at **{mean_h2['count']}/48 layers ({mean_h2['ranges']})**, although")
print("  the absolute separation remains very small.")
if max_h2['count'] == 0:
    print("- **Max pool:** no reliably positive situation separation at any layer.")
else:
    print(f"- **Max pool:** reliably positive at **{max_h2['count']}/48 layers ({max_h2['ranges']})**.")
print("- **Agent/recipient:** not evaluable in the aggregate analysis.\n")

print("These results show that there is **no single universal emergence layer** for")
print("situation-sensitive information. Instead, the onset and strength of separation")
print("depend on the representation being measured.\n")

print("### H1: Are Layers 23–26 Locally Privileged?\n")
print("The preregistered layers 23–26 showed a reliable local enhancement relative")
print("to neighboring layers for:\n")
print(f"- **Changed token:** Δ = {changed_h1['delta']:.5f},")
print(f"  95% CI [{changed_h1['ci_low']:.5f}, {changed_h1['ci_high']:.5f}], effect size = {changed_h1['effect_size']:.3f}")
print(f"- **Final token:** Δ = {final_h1['delta']:.5f},")
print(f"  95% CI [{final_h1['ci_low']:.5f}, {final_h1['ci_high']:.5f}], effect size = {final_h1['effect_size']:.3f}\n")

print("The **Event-token** and **Mean-pool** readouts did not show a reliable")
print("core-middle enhancement.\n")

print(f"Max pooling produced a statistically positive but extremely small local")
print(f"difference (Δ = {max_h1['delta']:.6f}). Because Max pooling showed no reliably positive")
print("situation separation at any layer, this effect is not interpreted as")
print("substantive evidence for a situation-sensitive core-middle representation.\n")

print("Overall, V4 provides **partial, readout-specific support for H1**. Layers")
print("23–26 are locally enhanced for Changed-token and Final-token representations,")
print("but situation-sensitive information is distributed much more broadly across")
print("the network and is not uniquely confined to this region.\n")

print("### H2: Does Situation Sensitivity Emerge Around Layers 8–12?\n")

print("V4 does not support a single universal onset around layers 8–12. Reliable")
print("separation begins at different depths depending on the readout: immediately")
print("for the Changed token, early for the Event and Final tokens, and later for")
print("Mean pooling.\n")

print("The V3-generated H2 should therefore be revised to a broader conclusion:\n")

print("> **Situation-sensitive information emerges and develops at different depths**")
print("> **depending on the representation being measured, rather than appearing at**")
print("> **one common transition point.**\n")

print("These results establish **representational geometry under strict held-out**")
print("**controls**. They do not establish that GPT-2 XL causally uses these")
print("representations.\n")

In [ ]:
# Calculate and display additional statistics: mean separation, peak separation, peak layer, core mean separation

readout_stats = []
for readout, g in curves.groupby('readout'):
    # Overall mean separation (across all layers)
    mean_separation = g['mean'].mean()

    # Peak separation and layer
    if not g['mean'].dropna().empty:
        peak_separation = g['mean'].max()
        peak_layer = g.loc[g['mean'].idxmax()]['layer']
    else:
        peak_separation = np.nan
        peak_layer = np.nan

    # Core mean separation (H1_CORE layers)
    core_mean_separation = g.query('layer in @H1_CORE')['mean'].mean()

    # Get H1 values from the previously calculated h1 DataFrame for CI and effect size
    h1_row = h1.query('readout == @readout').iloc[0] if not h1.query('readout == @readout').empty else None
    ci_low_h1 = h1_row['ci_low'] if h1_row is not None else np.nan
    ci_high_h1 = h1_row['ci_high'] if h1_row is not None else np.nan
    effect_size_h1 = h1_row['effect_size'] if h1_row is not None else np.nan
    core_minus_neighbors = h1_row['core_minus_neighbors'] if h1_row is not None else np.nan

    readout_stats.append({
        'readout': readout,
        'mean_separation': mean_separation,
        'peak_separation': peak_separation,
        'peak_layer': peak_layer,
        'core_mean_separation': core_mean_separation,
        'core_minus_neighbors': core_minus_neighbors,
        'ci_low_h1': ci_low_h1,
        'ci_high_h1': ci_high_h1,
        'effect_size_h1': effect_size_h1
    })

readout_stats_df = pd.DataFrame(readout_stats)
readout_stats_df = readout_stats_df.assign(readout=readout_stats_df.readout.map(LABELS))

print("\n### Additional Readout Statistics\n")
display(readout_stats_df.style.format({
    'mean_separation': '{:.5f}',
    'peak_separation': '{:.5f}',
    'peak_layer': '{:.0f}',
    'core_mean_separation': '{:.5f}',
    'core_minus_neighbors': '{:.5f}',
    'ci_low_h1': '{:.5f}',
    'ci_high_h1': '{:.5f}',
    'effect_size_h1': '{:.3f}'
}))

In [ ]:
def boot_mean(values):
    draws=values[rng.integers(0,len(values),size=(N_BOOT,len(values)))].mean(1); return values.mean(),np.quantile(draws,.025),np.quantile(draws,.975)
h1=[]
for readout,g in heldout.groupby('readout'):
    matrix=g.pivot(index='identifier',columns='layer',values='separation'); contrast=(matrix.loc[:,H1_CORE].mean(1)-matrix.loc[:,H1_NEIGHBORS].mean(1)).to_numpy(); mean,low,high=boot_mean(contrast); h1.append({'readout':readout,'core_minus_neighbors':mean,'ci_low':low,'ci_high':high,'supports_h1':low>0})
h1=pd.DataFrame(h1)
h2=[]
for readout,g in curves.groupby('readout'):
    positive=g.query('ci_low > 0').layer.to_list(); h2.append({'readout':readout,'first_reliably_positive_layer':min(positive) if positive else np.nan,'last_reliably_positive_layer':max(positive) if positive else np.nan,'positive_layer_count':len(positive)})
h2=pd.DataFrame(h2); h1.to_csv(OUT/'h1_results.csv',index=False); h2.to_csv(OUT/'h2_results.csv',index=False)
display(h1.assign(readout=h1.readout.map(LABELS)).round(4)); display(h2.assign(readout=h2.readout.map(LABELS)))
supported=h1[h1.supports_h1].readout.map(LABELS).to_list(); detected=h2.dropna(subset=['first_reliably_positive_layer'])
emergence='No prespecified readout showed reliable held-out separation.' if detected.empty else 'Reliable held-out separation: ' + '; '.join(f"{LABELS[r.readout]} at layers {int(r.first_reliably_positive_layer)}-{int(r.last_reliably_positive_layer)}" for r in detected.itertuples()) + '.'
print('V4 conclusion')
print(emergence)
print('H1 supported for: ' + ', '.join(supported) + '.' if supported else 'H1 was not supported by any prespecified readout.')
print('This is evidence about representational geometry under strict controls, not causal use.')



readout	core_minus_neighbors	ci_low	ci_high	supports_h1
0	Agent/recipient	NaN	NaN	NaN	False
1	Changed token	0.0070	0.0056	0.0085	True
2	Event token	-0.0003	-0.0009	0.0004	False
3	Final token	0.0018	0.0013	0.0023	True
4	Max pool	0.0000	0.0000	0.0000	True
5	Mean pool	0.0000	-0.0000	0.0001	False

readout	first_reliably_positive_layer	last_reliably_positive_layer	positive_layer_count
0	Agent/recipient	NaN	NaN	0
1	Changed token	0.0	47.0	48
2	Event token	4.0	47.0	38
3	Final token	1.0	47.0	43
4	Max pool	NaN	NaN	0
5	Mean pool	10.0	45.0	36
V4 conclusion
Reliable held-out separation: Changed token at layers 0-47; Event token at layers 4-47; Final token at layers 1-47; Mean pool at layers 10-45.
H1 supported for: Changed token, Final token, Max pool.
This is evidence about representational geometry under strict controls, not causal use.